# IT Service Ticket Classification — Fine-Tuning and Lightweight Optimization

## Introduction

This notebook fine-tunes `distilbert-base-uncased` for multi-class IT service ticket classification.

The goal is to run three controlled experiments using the same tokenized Hugging Face dataset created during Day 2. The experiments compare a baseline configuration with shorter and longer training schedules while keeping the batch size and learning rate stable.

This notebook prepares the outputs needed for Day 5, where the runs, metrics, and model artifacts will be tracked in MLflow.


## 1. Install Required Libraries

In [ ]:
# Install the Hugging Face Datasets library to load the tokenized dataset saved in Day 2.
!pip install datasets

# Install the Hugging Face Transformers library to load DistilBERT and use the Trainer API.
!pip install transformers

# Install the Evaluate library to calculate accuracy and F1-score.
!pip install evaluate

# Install scikit-learn to create classification reports and confusion matrices.
!pip install scikit-learn

# Install PyTorch, which is required to train transformer models.
!pip install torch

# Install pandas to organize experiment results in a table.
!pip install pandas

## 2. Import Libraries

In [ ]:
# Import os to work with folders and file paths.
import os

# Import json to load the label mappings created in Day 2.
import json

# Import numpy to convert model logits into predicted class IDs.
import numpy as np

# Import pandas to organize the final experiment metrics.
import pandas as pd

# Import torch to check whether GPU acceleration is available.
import torch

# Import load_from_disk to load the tokenized Hugging Face DatasetDict saved in Day 2.
from datasets import load_from_disk

# Import evaluate to calculate model evaluation metrics.
import evaluate

# Import AutoTokenizer to load the tokenizer used by DistilBERT.
from transformers import AutoTokenizer

# Import AutoModelForSequenceClassification to load DistilBERT with a classification head.
from transformers import AutoModelForSequenceClassification

# Import TrainingArguments to configure each training experiment.
from transformers import TrainingArguments

# Import Trainer to run the Hugging Face fine-tuning workflow.
from transformers import Trainer

# Import DataCollatorWithPadding to dynamically pad batches during training.
from transformers import DataCollatorWithPadding

# Import classification_report to inspect performance by class.
from sklearn.metrics import classification_report

# Import confusion_matrix to calculate the confusion matrix.
from sklearn.metrics import confusion_matrix

## 3. Define Project Paths

In [ ]:
# Define the path where the tokenized Hugging Face DatasetDict was saved in Day 2.
dataset_path = "../data/tokenized/tickets_distilbert"

# Define the path where the label-to-ID mapping was saved in Day 2.
label2id_path = "../data/metadata/label2id.json"

# Define the path where the ID-to-label mapping was saved in Day 2.
id2label_path = "../data/metadata/id2label.json"

# Define the base folder where all fine-tuned models will be saved.
models_base_path = "../models"

# Define the name of the pretrained model checkpoint used for transfer learning.
model_checkpoint = "distilbert-base-uncased

## 4. Load Tokenized Dataset and Label Mappings

In [ ]:
# Load the tokenized Hugging Face DatasetDict from disk.
dataset = load_from_disk(dataset_path)

# Display the dataset structure to confirm that train, validation, and test splits are available.
dataset

In [ ]:
# Open the label-to-ID JSON file created in Day 2.
with open(label2id_path, "r") as file:
    # Load the label-to-ID mapping into a Python dictionary.
    label2id = json.load(file)

# Open the ID-to-label JSON file created in Day 2.
with open(id2label_path, "r") as file:
    # Load the ID-to-label mapping into a Python dictionary.
    id2label = json.load(file)

# Convert JSON keys from strings to integers because model configuration expects integer label IDs.
id2label = {int(key): value for key, value in id2label.items()}

# Count the number of target classes in the classification problem.
num_labels = len(label2id)

# Display the number of classes found in the label mapping.
print("Number of labels:", num_labels)

# Display the label-to-ID mapping for verification.
label2id

In [ ]:
# Display the number of examples in the training split.
print("Train examples:", len(dataset["train"]))

# Display the number of examples in the validation split.
print("Validation examples:", len(dataset["validation"]))

# Display the number of examples in the test split.
print("Test examples:", len(dataset["test"]))

## 5. Load Tokenizer and Configure Device

In [ ]:
# Load the tokenizer associated with the selected checkpoint.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Create a data collator that dynamically pads each batch during training.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Check whether a GPU is available for faster training.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display the device that will be used by the training process.
print("Training device:", device)

## 6. Define Evaluation Metrics

In [ ]:
# Load the accuracy metric from the Evaluate library.
accuracy_metric = evaluate.load("accuracy")

# Load the F1-score metric from the Evaluate library.
f1_metric = evaluate.load("f1")

# Define a function that calculates evaluation metrics during validation.
def compute_metrics(eval_pred):
    # Unpack the model outputs and true labels from the evaluation prediction object.
    logits, labels = eval_pred

    # Convert logits into predicted class IDs by selecting the class with the highest score.
    predictions = np.argmax(logits, axis=-1)

    # Calculate accuracy using the predicted labels and true labels.
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]

    # Calculate macro F1-score to evaluate performance across all classes equally.
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    # Return the metrics to the Trainer.
    return {"accuracy": accuracy, "f1_macro": f1_macro}

## 7. Define Experiment Configurations

In [ ]:
# Define the list of fine-tuning experiments to run.
experiments = [
    # Define the baseline configuration from Day 3.
    {"run_name": "run_01_baseline_2_epochs", "model_dir": "distilbert_ticket_classifier_v1", "epochs": 2, "batch_size": 16, "learning_rate": 2e-5},

    # Define a shorter training configuration to compare whether one epoch is enough.
    {"run_name": "run_02_short_1_epoch", "model_dir": "distilbert_ticket_classifier_v2", "epochs": 1, "batch_size": 16, "learning_rate": 2e-5},

    # Define a longer training configuration to test whether one additional epoch improves results.
    {"run_name": "run_03_long_3_epochs", "model_dir": "distilbert_ticket_classifier_v3", "epochs": 3, "batch_size": 16, "learning_rate": 2e-5},
]

# Display the experiment plan before training.
pd.DataFrame(experiments)

## 8. Create Reusable Training Function

In [ ]:
# Define a function to create a fresh DistilBERT classification model for each experiment.
def load_fresh_model():
    # Load DistilBERT with a sequence classification head for the ticket categories.
    model = AutoModelForSequenceClassification.from_pretrained(
        # Use the pretrained DistilBERT checkpoint as the base model.
        model_checkpoint,
        # Set the number of target labels.
        num_labels=num_labels,
        # Attach the ID-to-label mapping to the model configuration.
        id2label=id2label,
        # Attach the label-to-ID mapping to the model configuration.
        label2id=label2id,
    )

    # Move the model to the selected training device.
    model.to(device)

    # Return the initialized model.
    return model

# Define a function that trains, evaluates, and saves one experiment.
def run_experiment(experiment):
    # Store the experiment run name for logging and folder naming.
    run_name = experiment["run_name"]

    # Build the output path where this experiment model will be saved.
    model_output_path = os.path.join(models_base_path, experiment["model_dir"])

    # Create a fresh model so each experiment starts from the same pretrained checkpoint.
    model = load_fresh_model()

    # Define the training configuration for this experiment.
    training_args = TrainingArguments(
        # Save training outputs in the model-specific output folder.
        output_dir=model_output_path,
        # Store the readable name of this experiment.
        run_name=run_name,
        # Set the learning rate for fine-tuning.
        learning_rate=experiment["learning_rate"],
        # Set the batch size used by each device during training.
        per_device_train_batch_size=experiment["batch_size"],
        # Set the batch size used by each device during evaluation.
        per_device_eval_batch_size=experiment["batch_size"],
        # Set the number of training epochs for this experiment.
        num_train_epochs=experiment["epochs"],
        # Evaluate the model at the end of each epoch.
        eval_strategy="epoch",
        # Disable automatic checkpoint saving to reduce disk usage.
        save_strategy="no",
        # Log training metrics at the end of each epoch.
        logging_strategy="epoch",
        # Disable external reporting integrations for this notebook.
        report_to="none",
        # Use mixed precision when CUDA is available to improve GPU training speed.
        fp16=torch.cuda.is_available(),
    )

    # Initialize the Hugging Face Trainer for this experiment.
    trainer = Trainer(
        # Pass the fresh model that will be fine-tuned.
        model=model,
        # Pass the training configuration.
        args=training_args,
        # Provide the tokenized training dataset.
        train_dataset=dataset["train"],
        # Provide the tokenized validation dataset.
        eval_dataset=dataset["validation"],
        # Provide the data collator for batch preparation.
        data_collator=data_collator,
        # Provide the metric function for validation evaluation.
        compute_metrics=compute_metrics,
    )

    # Print which experiment is starting.
    print(f"Starting experiment: {run_name}")

    # Start the fine-tuning process.
    trainer.train()

    # Evaluate the trained model on the validation split.
    validation_results = trainer.evaluate(eval_dataset=dataset["validation"])

    # Evaluate the trained model on the test split.
    test_results = trainer.evaluate(eval_dataset=dataset["test"])

    # Create the model output folder if it does not already exist.
    os.makedirs(model_output_path, exist_ok=True)

    # Save the fine-tuned model to disk.
    trainer.save_model(model_output_path)

    # Save the tokenizer with the model to make future inference easier.
    tokenizer.save_pretrained(model_output_path)

    # Store the main results in a compact dictionary.
    result = {
        # Store the run name.
        "run_name": run_name,
        # Store the model folder path.
        "model_path": model_output_path,
        # Store the number of epochs used.
        "epochs": experiment["epochs"],
        # Store the batch size used.
        "batch_size": experiment["batch_size"],
        # Store the learning rate used.
        "learning_rate": experiment["learning_rate"],
        # Store validation accuracy.
        "validation_accuracy": validation_results.get("eval_accuracy"),
        # Store validation macro F1-score.
        "validation_f1_macro": validation_results.get("eval_f1_macro"),
        # Store test accuracy.
        "test_accuracy": test_results.get("eval_accuracy"),
        # Store test macro F1-score.
        "test_f1_macro": test_results.get("eval_f1_macro"),
    }

    # Print where the model was saved.
    print("Model saved to:", model_output_path)

    # Return the trainer and result for later analysis.
    return trainer, result

## 9. Run Fine-Tuning Experiments

In [ ]:
# Create an empty list to store experiment results.
experiment_results = []

# Create an empty dictionary to keep each trained Trainer object available in memory.
trained_trainers = {}

# Loop through each experiment configuration.
for experiment in experiments:
    # Run the current fine-tuning experiment.
    trainer, result = run_experiment(experiment)

    # Store the trained Trainer object using the run name as the key.
    trained_trainers[experiment["run_name"]] = trainer

    # Append the compact metrics dictionary to the results list.
    experiment_results.append(result)

# Convert all experiment results into a DataFrame for comparison.
results_df = pd.DataFrame(experiment_results)

# Display the experiment comparison table.
results_df

## 10. Select Best Candidate by Validation Macro F1

In [ ]:
# Identify the row with the highest validation macro F1-score.
best_row = results_df.loc[results_df["validation_f1_macro"].idxmax()]

# Extract the best run name from the selected row.
best_run_name = best_row["run_name"]

# Extract the best model path from the selected row.
best_model_path = best_row["model_path"]

# Display the best run metadata without adding conclusions.
best_row

## 11. Generate Test Classification Report for the Best Candidate

In [ ]:
# Retrieve the Trainer object for the best run.
best_trainer = trained_trainers[best_run_name]

# Generate predictions on the test split using the best candidate model.
test_predictions_output = best_trainer.predict(dataset["test"])

# Convert model logits into predicted class IDs.
test_predictions = np.argmax(test_predictions_output.predictions, axis=-1)

# Extract the true labels from the prediction output.
test_labels = test_predictions_output.label_ids

# Create the ordered list of class names using the ID-to-label mapping.
target_names = [id2label[index] for index in sorted(id2label.keys())]

# Generate the test classification report as a dictionary.
report_dict = classification_report(test_labels, test_predictions, target_names=target_names, output_dict=True)

# Convert the classification report into a DataFrame for easier inspection.
classification_report_df = pd.DataFrame(report_dict).transpose()

# Display the classification report.
classification_report_df

## 12. Generate Test Confusion Matrix for the Best Candidate

In [ ]:
# Calculate the confusion matrix using true labels and predicted labels.
cm = confusion_matrix(test_labels, test_predictions)

# Convert the confusion matrix into a DataFrame with class names as rows and columns.
confusion_matrix_df = pd.DataFrame(cm, index=target_names, columns=target_names)

# Display the confusion matrix as a table.
confusion_matrix_df

## 13. Optional: Compress Saved Models for Download

In [ ]:
# Import shutil to compress model folders into zip files.
import shutil

# Loop through each experiment configuration.
for experiment in experiments:
    # Build the folder path of the saved model.
    model_folder = os.path.join(models_base_path, experiment["model_dir"])

    # Build the zip output path without the .zip extension.
    zip_base_path = os.path.join(models_base_path, experiment["model_dir"])

    # Compress the saved model folder into a zip file.
    shutil.make_archive(zip_base_path, "zip", model_folder)

    # Print the generated zip file path.
    print("Created:", zip_base_path + ".zip")